# Preprocessing notebook

- Stage 0 : videos + files verification
- Stage 1 : frame labels, clip index, and splits
- Stage 2 : embedding extraction / cache verification

In [ ]:
# standard library imports
from pathlib import Path
import json
import os
import random
import subprocess
import time
import math

# data processing and visualization
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt

# deep learning
import torch
import torch.nn as nn

# notebook display
from IPython.display import display, Image

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()  # main project folder

# datasets
DATASETS_DIR = PROJECT_ROOT / "datasets"

CHOLEC80_ROOT = DATASETS_DIR / "Cholec80"
VIDEOS_DIR = CHOLEC80_ROOT / "videos"

SMOKE_ROOT = DATASETS_DIR / "Smoke_Cholec80"
SMOKE_MAP_DIR = SMOKE_ROOT / "cholec80_mappings"

# artifacts
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ART_COMMON = ARTIFACTS_DIR / "common"
ART_INDICES = ARTIFACTS_DIR / "indices"
ART_FIGS = ARTIFACTS_DIR / "figures"
ART_LOGS = ARTIFACTS_DIR / "logs"

# features/embeddings
ART_FEATURES = ARTIFACTS_DIR / "features"
ART_EMBEDDINGS = ARTIFACTS_DIR / "embeddings"

# create output folders if they dont exist
for d in [ARTIFACTS_DIR, ART_COMMON, ART_INDICES, ART_FIGS, ART_LOGS, ART_EMBEDDINGS]:
    d.mkdir(parents=True, exist_ok=True)

# reproducibility
RNG_SEED = 67
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

# dataset availability
CHOLEC80_AVAILABLE = CHOLEC80_ROOT.exists() and VIDEOS_DIR.exists()
SMOKE_AVAILABLE = SMOKE_ROOT.exists() and SMOKE_MAP_DIR.exists()

if not CHOLEC80_AVAILABLE:
    print("Cholec80 available  : False (dataset not found, continuing anyway)")
else:
    print("Cholec80 available  : True")

assert PROJECT_ROOT.exists(), f"Missing project root : {PROJECT_ROOT}"
assert SMOKE_AVAILABLE, f"Missing Smoke_Cholec80 dataset or mappings folder : {SMOKE_MAP_DIR}"

print("Project folder         :", PROJECT_ROOT)
print("Datasets folder        :", DATASETS_DIR)
print("Cholec80 folder        :", CHOLEC80_ROOT)
print("Videos folder          :", VIDEOS_DIR)
print("Smoke dataset folder   :", SMOKE_ROOT)
print("Smoke mappings folder  :", SMOKE_MAP_DIR)
print("\nCholec80 available     :", CHOLEC80_AVAILABLE)

In [ ]:
# finalize and save plot
def finalize_plot(title, xlabel, ylabel, out_path: Path, show=True):
    plt.title(title, pad=12, fontweight="bold")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(axis="y", linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.savefig(out_path, dpi=220, bbox_inches="tight")

    if show:
        plt.show()

    plt.close()


# save histogram
def save_hist(data, title, xlabel, ylabel, out_path: Path,
              bins=40, color="#4C78A8", show=True):
    plt.figure(figsize=(10, 6), dpi=120)
    plt.hist(
        data,
        bins=bins,
        color=color,
        edgecolor="black",
        linewidth=0.8
    )
    finalize_plot(title, xlabel, ylabel, out_path, show=show)


# save bar chart
def save_bar(categories, counts, title, xlabel, ylabel, out_path: Path,
             rotate=20, show_values=True, show=True):
    plt.figure(figsize=(10, 6), dpi=120)

    bars = plt.bar(
        categories,
        counts,
        edgecolor="black",
        linewidth=0.8
    )

    plt.xticks(rotation=rotate, ha="right")

    if show_values:
        for bar, c in zip(bars, counts):
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{int(c)}",
                ha="center",
                va="bottom",
                fontsize=10
            )

    finalize_plot(title, xlabel, ylabel, out_path, show=show)


# save line plot
def save_line(x, y, title, xlabel, ylabel, out_path: Path,
              color="#4C78A8", linewidth=1.5, show=True):
    plt.figure(figsize=(12, 4), dpi=120)
    plt.plot(x, y, color=color, linewidth=linewidth)
    finalize_plot(title, xlabel, ylabel, out_path, show=show)


# simple helper for figure subfolders
def ensure_fig_dir(name: str) -> Path:
    p = ART_FIGS / name
    p.mkdir(parents=True, exist_ok=True)
    return p

In [ ]:
# ffmpeg check
try:
    out = subprocess.check_output(["ffmpeg", "-version"], text=True)
    print("ffmpeg :", out.splitlines()[0])
except Exception as e:
    print("ffmpeg check skipped/failed :", repr(e))

# torch + device check
try:
    if torch.backends.mps.is_available():
        DEVICE = torch.device("mps")
    elif torch.cuda.is_available():
        DEVICE = torch.device("cuda")
    else:
        DEVICE = torch.device("cpu")

    print("torch  :", torch.__version__)
    print("device :", DEVICE)
except Exception as e:
    print("torch/device check skipped/failed :", repr(e))
    DEVICE = torch.device("cpu")

## Stage 0 - Data verification

In [ ]:
# expected dataset stats 
FPS_EXPECTED = 25.0
FPS_TOL = 0.5 # small fps tolerance 

VIDEO_INV_CSV = ART_COMMON / "video_inventory.csv"
SMOKE_SEG_CSV = ART_COMMON / "smoke_segment_inventory.csv"

# open a video safely
def safe_video_capture(video_path: Path) -> cv2.VideoCapture:
    if not video_path.exists():
        raise FileNotFoundError(f"Video file does not exist : {video_path}")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        raise RuntimeError(f"Cannot open video : {video_path}")

    return cap


# get basic video metadata
def get_video_meta(video_path: Path):
    cap = safe_video_capture(video_path)
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    duration_sec = frame_count / fps if fps > 0 else None
    return fps, frame_count, duration_sec


# read one frame safely
def read_frame_at(video_path: Path, frame_idx: int):
    cap = safe_video_capture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    cap.release()
    return ok, frame

In [ ]:
# first build an inventory of all videos 

ART_FIG0 = ensure_fig_dir("stage0")

if CHOLEC80_AVAILABLE:
    video_files = sorted([p for p in VIDEOS_DIR.glob("*.mp4") if not p.name.startswith("._")])

    if len(video_files) == 0:
        raise RuntimeError(f"No .mp4 files found in {VIDEOS_DIR}")

    print(f"Found {len(video_files)} videos.")
    if len(video_files) != 80:
        print("Expected 80 videos, found :", len(video_files))

    rows = []
    bad_open = []
    bad_fps = []
    frame_read_failures = []

    random.seed(RNG_SEED)

    for vp in tqdm(video_files, desc="Scanning videos"):
        vid = vp.stem

        try:
            fps, frame_count, duration_sec = get_video_meta(vp)
        except Exception as e:
            bad_open.append((vid, str(vp), repr(e)))
            continue

        if abs(fps - FPS_EXPECTED) > FPS_TOL:
            bad_fps.append((vid, fps))

        # quick random frame check
        if frame_count > 0:
            probe_idx = random.randint(0, max(0, frame_count - 1))
            ok, _ = read_frame_at(vp, probe_idx)
            if not ok:
                frame_read_failures.append((vid, probe_idx))

        rows.append({
            "video_id": vid,
            "filename": vp.name,
            "path": str(vp),
            "fps": fps,
            "frame_count": frame_count,
            "duration_sec": duration_sec
        })

    videos_df = pd.DataFrame(rows).sort_values("video_id").reset_index(drop=True)
    videos_df.to_csv(VIDEO_INV_CSV, index=False)

    print("Saved video inventory :", VIDEO_INV_CSV)
    display(videos_df.head(5))

    if bad_open:
        pd.DataFrame(bad_open, columns=["video_id", "path", "error"]).to_csv(
            ART_LOGS / "stage0_bad_open.csv", index=False
        )
        raise RuntimeError("Some videos failed to open. See stage0_bad_open.csv")

    if bad_fps:
        pd.DataFrame(bad_fps, columns=["video_id", "fps"]).to_csv(
            ART_LOGS / "stage0_bad_fps.csv", index=False
        )
        raise RuntimeError("Some videos have unexpected fps. See stage0_bad_fps.csv")

    if frame_read_failures:
        pd.DataFrame(frame_read_failures, columns=["video_id", "frame_idx"]).to_csv(
            ART_LOGS / "stage0_bad_frame_reads.csv", index=False
        )
        raise RuntimeError("Some videos had frame read failures. See stage0_bad_frame_reads.csv")

else:
    print("Cholec80 is not currently mounted.")

    if not VIDEO_INV_CSV.exists():
        raise RuntimeError(
            "No live Cholec80 dataset and no cached video_inventory.csv found.\n"
            "Connect Cholec80 once and run stage 0 again."
        )

    print(f"Loading cached video inventory : {VIDEO_INV_CSV}")
    videos_df = pd.read_csv(VIDEO_INV_CSV)

    required_cols = {"video_id", "filename", "path", "fps", "frame_count", "duration_sec"}
    if not required_cols.issubset(set(videos_df.columns)):
        raise RuntimeError(f"Cached video inventory is missing columns. Found : {videos_df.columns.tolist()}")

    display(videos_df.head(5))

In [ ]:
save_hist(
    data=videos_df["fps"].values,
    title="Cholec80 FPS distribution",
    xlabel="fps",
    ylabel="count",
    out_path=ART_FIG0 / "fps_hist.png",
    bins=20
)

save_hist(
    data=videos_df["duration_sec"].values,
    title="Video duration distribution",
    xlabel="duration (seconds)",
    ylabel="count",
    out_path=ART_FIG0 / "video_duration_hist.png",
    bins=30
)

print("Saved stage 0 video plots.")

In [ ]:
# load smoke annotation csvs and normalize them into a cleaner table 

index_csv = SMOKE_MAP_DIR / "index.csv"
labels_csv = SMOKE_MAP_DIR / "labels.csv"

if not index_csv.exists():
    raise RuntimeError(f"Missing Smoke index.csv : {index_csv}")
if not labels_csv.exists():
    raise RuntimeError(f"Missing Smoke labels.csv : {labels_csv}")

index_df = pd.read_csv(index_csv)
label_map_df = pd.read_csv(labels_csv)

print("Smoke index.csv head :")
display(index_df.head(5))

print("Smoke label map :")
display(label_map_df)

seg_files = sorted([p for p in SMOKE_MAP_DIR.glob("*.mp4.csv") if not p.name.startswith("._")])
if len(seg_files) == 0:
    raise RuntimeError(f"No video smoke segment CSV files found in {SMOKE_MAP_DIR}")

print(f"Found {len(seg_files)} smoke segment files.")
if len(seg_files) != 80:
    print("Expected 80 smoke csv files, found :", len(seg_files))

required_cols = {"frame_start", "frame_end", "label"}
all_segs = []
bad_seg_files = []

for sf in tqdm(seg_files, desc="Reading smoke segments"):
    try:
        df = pd.read_csv(sf)
    except Exception as e:
        bad_seg_files.append((sf.name, repr(e)))
        continue

    if not required_cols.issubset(df.columns):
        bad_seg_files.append((sf.name, f"Missing columns {df.columns.tolist()}"))
        continue

    vid = sf.name.replace(".mp4.csv", "")
    df["video_id"] = vid

    for c in ["frame_start", "frame_end", "label"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=["frame_start", "frame_end", "label"])
    if len(df) == 0:
        continue

    df = df.astype({"frame_start": int, "frame_end": int, "label": int})

    df = df[
        (df["label"].isin([0, 1])) &
        (df["frame_start"] >= 0) &
        (df["frame_end"] >= 0) &
        (df["frame_start"] <= df["frame_end"])
    ].copy()

    if len(df) == 0:
        continue

    df["len_frames"] = df["frame_end"] - df["frame_start"] + 1
    df["len_seconds"] = df["len_frames"] / FPS_EXPECTED
    all_segs.append(df)

if bad_seg_files:
    pd.DataFrame(bad_seg_files, columns=["file", "error"]).to_csv(
        ART_LOGS / "stage0_bad_segment_files.csv", index=False
    )
    print("Some segment files had issues. See :", ART_LOGS / "stage0_bad_segment_files.csv")

if len(all_segs) == 0:
    raise RuntimeError("No valid smoke segments were collected after validation.")

smoke_segments_df = pd.concat(all_segs, ignore_index=True)
smoke_segments_df.to_csv(SMOKE_SEG_CSV, index=False)

print("Saved smoke segment inventory :", SMOKE_SEG_CSV)
display(smoke_segments_df.head(10))

In [ ]:
# check smoke segments stay inside video frame ranges
video_frames_map = dict(zip(videos_df["video_id"], videos_df["frame_count"].astype(int)))

violations = []
missing_video_ids = []

for r in smoke_segments_df.itertuples(index=False):
    vid = r.video_id

    if vid not in video_frames_map:
        missing_video_ids.append(vid)
        continue

    nframes = int(video_frames_map[vid])

    if r.frame_start < 0 or r.frame_end < 0 or r.frame_end >= nframes:
        violations.append({
            "video_id": vid,
            "frame_start": int(r.frame_start),
            "frame_end": int(r.frame_end),
            "frame_count": nframes,
            "label": int(r.label)
        })

if missing_video_ids:
    pd.DataFrame({"video_id": sorted(set(missing_video_ids))}).to_csv(
        ART_LOGS / "stage0_missing_video_ids.csv", index=False
    )
    raise RuntimeError("Smoke mapping references missing video ids. See stage0_missing_video_ids.csv")

if violations:
    pd.DataFrame(violations).to_csv(
        ART_LOGS / "stage0_out_of_range_segments.csv", index=False
    )
    raise RuntimeError("Some smoke segments are outside video frame range. See stage0_out_of_range_segments.csv")

# smoke-only stats
sm1 = smoke_segments_df[smoke_segments_df["label"] == 1].copy()

save_hist(
    data=sm1["len_frames"].values,
    title="Smoke segment length distribution (frames)",
    xlabel="segment length (frames)",
    ylabel="count",
    out_path=ART_FIG0 / "smoke_segment_length_hist.png",
    bins=60
)

smoke_seconds = (
    sm1.groupby("video_id")["len_seconds"]
    .sum()
    .reset_index(name="smoke_seconds_total")
)

save_hist(
    data=smoke_seconds["smoke_seconds_total"].values,
    title="Total smoke duration per video",
    xlabel="smoke duration (seconds)",
    ylabel="count",
    out_path=ART_FIG0 / "smoke_total_seconds_per_video_hist.png",
    bins=25
)

smoke_counts = (
    sm1.groupby("video_id")
    .size()
    .reset_index(name="smoke_segment_count")
)

save_hist(
    data=smoke_counts["smoke_segment_count"].values,
    title="Smoke segment count per video",
    xlabel="number of smoke segments",
    ylabel="count",
    out_path=ART_FIG0 / "smoke_segment_count_per_video_hist.png",
    bins=20
)

print("Stage 0 smoke checks passed.")

In [ ]:
# export frames around onset for smoke segments from video60 - 1s before onset, onset, 1s after onset
# then create a contact sheet 

sheet_path = align_dir / "alignment_contact_sheet_video60.png"

align_dir = ensure_fig_dir("alignment_samples")
random.seed(RNG_SEED)

saved_imgs = []

chosen_vid = "video60"
video_path = VIDEOS_DIR / f"{chosen_vid}.mp4"

if not video_path.exists():
    print("Missing video file for", chosen_vid, "->", video_path)
    if sheet_path.exists():
        display(Image(filename=str(sheet_path)))
    else:
        print("No previously saved contact sheet found.")

else:
    # use the smoke-only dataframe created earlier
    vid_segs = sm1[sm1["video_id"] == chosen_vid].copy()

    if len(vid_segs) == 0:
        print("No smoke segments found for", chosen_vid)

    else:
        # keep few segments only so the sheet stays readable
        segs = vid_segs.sample(
            n=min(6, len(vid_segs)),
            random_state=RNG_SEED
        )

        for i, r in enumerate(segs.itertuples(index=False)):
            S = int(r.frame_start)

            # 1 second before, onset, 1 second after
            offsets = [-25, 0, 25]

            for off in offsets:
                fi = max(0, S + off)

                ok, frame = read_frame_at(video_path, fi)
                if not ok or frame is None:
                    print(f"Failed reading {chosen_vid} frame {fi}")
                    continue

                frame = cv2.resize(frame, (320, 180), interpolation=cv2.INTER_AREA)

                phase = "before" if off < 0 else ("onset" if off == 0 else "after")

                cv2.putText(frame, chosen_vid, (12, 28),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                            (255, 255, 255), 2, cv2.LINE_AA)

                cv2.putText(frame, f"seg{i} | {phase} | frame={fi}", (12, 54),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                            (255, 255, 255), 2, cv2.LINE_AA)

                out_path = align_dir / f"{chosen_vid}_seg{i}_fi{fi}_onset{S}_off{off}.jpg"
                cv2.imwrite(str(out_path), frame)
                saved_imgs.append(out_path)

        print("Alignment images saved in :", align_dir)
        print("Count :", len(saved_imgs))

        # build contact sheet
        if len(saved_imgs) > 0:
            cols = 3
            imgs = []

            for p in saved_imgs:
                im = cv2.imread(str(p))
                if im is None:
                    continue

                im = cv2.resize(im, (320, 180), interpolation=cv2.INTER_AREA)
                imgs.append(im)

            if len(imgs) > 0:
                rows = (len(imgs) + cols - 1) // cols

                while len(imgs) < rows * cols:
                    imgs.append(np.zeros_like(imgs[0]))

                grid_rows = []
                for r in range(rows):
                    row_imgs = imgs[r * cols:(r + 1) * cols]
                    row = np.hstack(row_imgs)
                    grid_rows.append(row)

                sheet = np.vstack(grid_rows)

                sheet_path = align_dir / "alignment_contact_sheet_video60.png"
                cv2.imwrite(str(sheet_path), sheet)

                print("Saved contact sheet ->", sheet_path)
                display(Image(filename=str(sheet_path)))

            else:
                print("Contact sheet skipped (no images could be loaded).")

        else:
            print("Contact sheet skipped (no images were saved).")

## Stage 1 - Labels and split construction

In [ ]:
SEED = 67
FPS = 25.0

# forecast horizon W
W_SECONDS = 2.0 #try 0.5, 1.0, 2.0, 3.0
W_FRAMES = int(round(W_SECONDS * FPS))

# input history length L
CLIP_LEN_SECONDS = 2.0
CLIP_LEN_FRAMES = int(round(CLIP_LEN_SECONDS * FPS))

# stride between clips
STRIDE_SECONDS = 0.2
STRIDE_FRAMES = int(round(STRIDE_SECONDS * FPS))

# split ratios by video
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# timeline labels
LABEL_CLEAR = 0
LABEL_IMMINENT = 1
LABEL_ONSET = 2
LABEL_SMOKE = 3

# for horizon experiments 
def make_run_name(w_sec, clip_sec, stride_sec):
    a = str(w_sec).replace(".", "p")
    b = str(clip_sec).replace(".", "p")
    c = str(stride_sec).replace(".", "p")
    return f"w{a}s_l{b}s_s{c}s"

RUN_NAME = make_run_name(W_SECONDS, CLIP_LEN_SECONDS, STRIDE_SECONDS)

RUN_DIR = ART_INDICES / RUN_NAME
LABELS_DIR = RUN_DIR / "labels_npz"
RUN_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

STAGE1_CONFIG = {
    "seed": SEED,
    "fps": FPS,
    "w_seconds": W_SECONDS,
    "w_frames": W_FRAMES,
    "clip_len_seconds": CLIP_LEN_SECONDS,
    "clip_len_frames": CLIP_LEN_FRAMES,
    "stride_seconds": STRIDE_SECONDS,
    "stride_frames": STRIDE_FRAMES,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "label_encoding": {
        "clear": LABEL_CLEAR,
        "imminent": LABEL_IMMINENT,
        "onset": LABEL_ONSET,
        "smoke": LABEL_SMOKE
    }
}

(RUN_DIR / "config.json").write_text(json.dumps(STAGE1_CONFIG, indent=2))

print("Run name      :", RUN_NAME)
print("Run folder    :", RUN_DIR)
print("W frames      :", W_FRAMES)
print("Clip len      :", CLIP_LEN_FRAMES)
print("Stride frames :", STRIDE_FRAMES)

In [ ]:
# reload saved inventories 
# few checks for bad metadata 

videos_df = pd.read_csv(VIDEO_INV_CSV)
smoke_segments_df = pd.read_csv(SMOKE_SEG_CSV)

# basic clean
videos_df = videos_df[~videos_df["filename"].astype(str).str.startswith("._")].copy()
videos_df = videos_df.sort_values("video_id").reset_index(drop=True)

if len(videos_df) != 80:
    print(f"Expected 80 videos, got {len(videos_df)}")

fps_bad = videos_df[np.abs(videos_df["fps"].astype(float) - FPS) > 0.5]
if len(fps_bad) > 0:
    raise RuntimeError(f"Found videos with unexpected fps :\n{fps_bad[['video_id', 'fps']].head(20)}")

required_smoke_cols = {"video_id", "frame_start", "frame_end", "label"}
if not required_smoke_cols.issubset(set(smoke_segments_df.columns)):
    raise RuntimeError(f"smoke_segment_inventory.csv missing columns. Found : {smoke_segments_df.columns.tolist()}")

for c in ["frame_start", "frame_end", "label"]:
    smoke_segments_df[c] = pd.to_numeric(smoke_segments_df[c], errors="coerce")

bad_nan = smoke_segments_df[smoke_segments_df[["frame_start", "frame_end", "label"]].isna().any(axis=1)]
if len(bad_nan) > 0:
    out_bad = ART_LOGS / "stage1_bad_nan_smoke_segments.csv"
    bad_nan.to_csv(out_bad, index=False)
    raise RuntimeError(f"NaNs found in smoke segments. Saved : {out_bad}")

smoke_segments_df["frame_start"] = smoke_segments_df["frame_start"].astype(int)
smoke_segments_df["frame_end"] = smoke_segments_df["frame_end"].astype(int)
smoke_segments_df["label"] = smoke_segments_df["label"].astype(int)

# videos split
rng = random.Random(SEED)
video_ids = videos_df["video_id"].tolist()
rng.shuffle(video_ids)

n = len(video_ids)
n_train = int(round(n * TRAIN_RATIO))
n_val = int(round(n * VAL_RATIO))
n_test = n - n_train - n_val

train_vids = sorted(video_ids[:n_train])
val_vids = sorted(video_ids[n_train:n_train + n_val])
test_vids = sorted(video_ids[n_train + n_val:])

assert len(set(train_vids) & set(val_vids)) == 0
assert len(set(train_vids) & set(test_vids)) == 0
assert len(set(val_vids) & set(test_vids)) == 0
assert len(train_vids) + len(val_vids) + len(test_vids) == n

splits = {
    "seed": SEED,
    "run_name": RUN_NAME,
    "ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
    "counts": {"train": len(train_vids), "val": len(val_vids), "test": len(test_vids)},
    "videos": {"train": train_vids, "val": val_vids, "test": test_vids}
}

splits_path = RUN_DIR / "splits.json"
splits_path.write_text(json.dumps(splits, indent=2))

print("Saved splits :", splits_path)
print("Split counts :", splits["counts"])

# smoke coverage by split
def smoke_count_for(vid_list):
    return int(((smoke_segments_df["video_id"].isin(vid_list)) & (smoke_segments_df["label"] == 1)).sum())

print("Smoke rows in train :", smoke_count_for(train_vids))
print("Smoke rows in val   :", smoke_count_for(val_vids))
print("Smoke rows in test  :", smoke_count_for(test_vids))

In [ ]:
# convert smoke segments into frame-wise labels for each video
# label meanings : 0 = clear, 1 = imminent, 2 = onset

frame_count_map = dict(zip(videos_df["video_id"], videos_df["frame_count"].astype(int)))

frame_count_map = dict(zip(videos_df["video_id"], videos_df["frame_count"].astype(int)))

def build_labels_for_video(video_id: str, frame_count: int, seg_df: pd.DataFrame, w_frames: int) -> np.ndarray:
    labels = np.zeros((frame_count,), dtype=np.uint8)

    smoke_segs = seg_df[(seg_df["video_id"] == video_id) & (seg_df["label"] == 1)].copy()
    smoke_segs = smoke_segs.sort_values("frame_start")

    for r in smoke_segs.itertuples(index=False):
        s = int(r.frame_start)
        e = int(r.frame_end)

        if frame_count <= 0:
            continue

        # clamp to valid range
        s = max(0, min(s, frame_count - 1))
        e = max(0, min(e, frame_count - 1))
        if e < s:
            continue

        # only the first smoke frame is onset
        labels[s] = LABEL_ONSET

        # all later smoke frames become ongoing smoke
        if e > s:
            labels[s + 1:e + 1] = LABEL_SMOKE

        # imminent region = W frames before smoke starts
        i0 = max(0, s - w_frames)
        i1 = max(0, s - 1)

        if i1 >= i0:
            sl = labels[i0:i1 + 1]

            # only overwrite clear frames, never overwrite onset/smoke
            sl[sl == LABEL_CLEAR] = LABEL_IMMINENT
            labels[i0:i1 + 1] = sl

    return labels


label_summary_rows = []

for vid in tqdm(videos_df["video_id"].tolist(), desc="Building timeline labels"):
    nframes = int(frame_count_map[vid])
    labels = build_labels_for_video(vid, nframes, smoke_segments_df, W_FRAMES)

    np.savez_compressed(LABELS_DIR / f"{vid}.npz", labels=labels)

    counts = np.bincount(labels, minlength=4)
    label_summary_rows.append({
        "video_id": vid,
        "frame_count": nframes,
        "clear_frames": int(counts[LABEL_CLEAR]),
        "imminent_frames": int(counts[LABEL_IMMINENT]),
        "onset_frames": int(counts[LABEL_ONSET]),
        "smoke_frames": int(counts[LABEL_SMOKE])
    })

label_summary_df = pd.DataFrame(label_summary_rows).sort_values("video_id").reset_index(drop=True)
label_summary_path = RUN_DIR / "label_summary.csv"
label_summary_df.to_csv(label_summary_path, index=False)

print("Saved timeline labels to :", LABELS_DIR)
print("Saved label summary to   :", label_summary_path)
display(label_summary_df.head(5))

In [ ]:
sample_vid = videos_df["video_id"].iloc[0]
labels = np.load(LABELS_DIR / f"{sample_vid}.npz")["labels"]

unique, counts = np.unique(labels, return_counts=True)
print("video:", sample_vid)
print(dict(zip(unique, counts)))

print("first 200 labels:")
print(labels[:1000].tolist())

In [ ]:
# roll frame labels into clip-level training rows
# each row points to a clip window plus the target for the last frame in that window

def build_clip_index_for_video(video_id: str, labels: np.ndarray, clip_len: int, stride: int, w_frames: int):
    rows = []
    n = len(labels)

    # need enough past for the clip and enough future for forecasting
    t_min = clip_len - 1
    t_max = n - w_frames - 1

    if t_max < t_min:
        return rows

    for t in range(t_min, t_max + 1, stride):
        clip_start = t - clip_len + 1
        clip_end = t

        look_start = t + 1
        look_end = t + w_frames

        # do not allow smoke already inside the input clip
        history = labels[clip_start:clip_end + 1]
        if np.any((history == LABEL_ONSET) | (history == LABEL_SMOKE)):
            continue

        # target = will onset happen in the next W frames
        will_onset = int(np.any(labels[look_start:look_end + 1] == LABEL_ONSET))

        # distance to next onset from t+1
        next_idx = np.where(labels[look_start:] == LABEL_ONSET)[0]
        dist_to_next_onset = int(next_idx[0]) if len(next_idx) > 0 else -1

        rows.append({
            "video_id": video_id,
            "t": int(t),
            "clip_start": int(clip_start),
            "clip_end": int(clip_end),
            "target": int(will_onset),
            "dist_to_next_onset_from_lookstart": dist_to_next_onset
        })

    return rows


all_clip_rows = []

for vid in tqdm(videos_df["video_id"].tolist(), desc="Building clip index"):
    labels = np.load(LABELS_DIR / f"{vid}.npz")["labels"]
    rows = build_clip_index_for_video(
        video_id=vid,
        labels=labels,
        clip_len=CLIP_LEN_FRAMES,
        stride=STRIDE_FRAMES,
        w_frames=W_FRAMES
    )
    all_clip_rows.extend(rows)

clips_index = pd.DataFrame(all_clip_rows)

def assign_split(video_id: str):
    if video_id in train_vids:
        return "train"
    if video_id in val_vids:
        return "val"
    return "test"

clips_index["split"] = clips_index["video_id"].map(assign_split)

clips_csv = RUN_DIR / "clips_index.csv"
clips_index.to_csv(clips_csv, index=False)

print("Saved clip index :", clips_csv)
print("Total clips      :", len(clips_index))
display(clips_index.head(5))

In [ ]:
ART_FIG1 = ensure_fig_dir(RUN_NAME)

# class balance summary
def split_stats(df, split_name):
    sub = df[df["split"] == split_name]
    pos_rate = float(sub["target"].mean()) if len(sub) > 0 else 0.0
    return {
        "rows": int(len(sub)),
        "pos_count": int(sub["target"].sum()),
        "pos_rate": pos_rate
    }

stats = {
    "overall": {
        "rows": int(len(clips_index)),
        "pos_count": int(clips_index["target"].sum()),
        "pos_rate": float(clips_index["target"].mean())
    },
    "train": split_stats(clips_index, "train"),
    "val": split_stats(clips_index, "val"),
    "test": split_stats(clips_index, "test"),
    "params": STAGE1_CONFIG
}

class_balance_path = RUN_DIR / "class_balance.json"
class_balance_path.write_text(json.dumps(stats, indent=2))

print("Saved class balance :", class_balance_path)
print("Positive rates :", {k: round(stats[k]["pos_rate"], 4) for k in ["train", "val", "test"]})

# plot 1 - forecasting target balance
save_bar(
    categories=["0", "1"],
    counts=[
        int((clips_index["target"] == 0).sum()),
        int((clips_index["target"] == 1).sum())
    ],
    title=f"Forecasting target balance ({RUN_NAME})",
    xlabel="target",
    ylabel="count",
    out_path=ART_FIG1 / "target_balance.png",
    rotate=0
)

# plot 2 - positive rate by split
save_bar(
    categories=["train", "val", "test"],
    counts=[
        stats["train"]["pos_rate"] * 100,
        stats["val"]["pos_rate"] * 100,
        stats["test"]["pos_rate"] * 100
    ],
    title=f"Positive rate by split ({RUN_NAME})",
    xlabel="split",
    ylabel="positive rate (%)",
    out_path=ART_FIG1 / "positive_rate_by_split.png",
    rotate=0,
    show_values=False
)

# plot 3 - label timeline example
example_vid = videos_df["video_id"].iloc[0]
example_labels = np.load(LABELS_DIR / f"{example_vid}.npz")["labels"]

plt.figure(figsize=(14, 3), dpi=120)
plt.plot(np.arange(len(example_labels)), example_labels, linewidth=1.0)
plt.yticks([0, 1, 2, 3], ["clear", "imminent", "onset", "smoke"])
plt.xlabel("frame index")
plt.ylabel("timeline label")
plt.title(f"Label timeline example - {example_vid} ({RUN_NAME})", pad=12, fontweight="bold")
plt.tight_layout()
plt.savefig(ART_FIG1 / f"label_timeline_{example_vid}.png", dpi=220, bbox_inches="tight")
plt.show()
plt.close()

# plot 4 - distance to onset sanity check for positive clips
sample_n = min(50000, len(clips_index))
sample = clips_index.sample(n=sample_n, random_state=SEED).copy()
pos = sample[(sample["target"] == 1) & (sample["dist_to_next_onset_from_lookstart"] >= 0)].copy()

save_hist(
    data=pos["dist_to_next_onset_from_lookstart"].values,
    title=f"Distance to next onset for positive clips ({RUN_NAME})",
    xlabel="frames from look_start to next onset",
    ylabel="count",
    out_path=ART_FIG1 / "positive_clip_onset_distance_hist.png",
    bins=min(40, max(10, W_FRAMES))
)

print("Stage 1 complete.")

## Stage 2 - Embeddings cache build / verification

In [ ]:
# stage 2 config

IMG_SIZE = 224
BATCH_SIZE = 64

# use stage 1 output from the selected run
CLIPS_CSV = RUN_DIR / "clips_index.csv"
assert CLIPS_CSV.exists(), f"Missing clip index : {CLIPS_CSV}"

EFF_DIR = ART_EMBEDDINGS / "effb0"
CONVNEXT_DIR = ART_EMBEDDINGS / "convnext_tiny"

EFF_DIR.mkdir(parents=True, exist_ok=True)
CONVNEXT_DIR.mkdir(parents=True, exist_ok=True)

print("EfficientNet embeddings folder :", EFF_DIR)
print("ConvNeXt embeddings folder     :", CONVNEXT_DIR)

clips_index = pd.read_csv(CLIPS_CSV)
videos_df = pd.read_csv(VIDEO_INV_CSV).sort_values("video_id").reset_index(drop=True)

print("clips rows :", len(clips_index))
print("videos     :", len(videos_df))
display(clips_index.head(3))

In [ ]:
# read a whole clip if needed later for debugging or quick checks
def safe_open_video(video_path: Path) -> cv2.VideoCapture:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        raise RuntimeError(f"Cannot open video : {video_path}")
    return cap


def read_clip_bgr(video_path: Path, start_f: int, end_f: int, resize_hw=(224, 224)):
    if end_f < start_f:
        return False, None

    t = end_f - start_f + 1
    cap = safe_open_video(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_f))

    frames = np.zeros((t, resize_hw[1], resize_hw[0], 3), dtype=np.uint8)
    got = 0

    for i in range(t):
        ok, fr = cap.read()
        if not ok or fr is None:
            break

        fr = cv2.resize(fr, resize_hw, interpolation=cv2.INTER_AREA)
        frames[i] = fr
        got += 1

    cap.release()

    if got != t:
        return False, frames[:got]

    return True, frames


def bgr_batch_to_torch_rgb(batch_bgr, mean, std):
    # batch_bgr : [B,H,W,3] uint8 in bgr
    # returns   : [B,3,H,W] float tensor in normalized rgb
    x = batch_bgr[..., ::-1].copy()  # bgr -> rgb
    x = x.astype(np.float32) / 255.0
    x = np.transpose(x, (0, 3, 1, 2))

    mean = np.asarray(mean, dtype=np.float32).reshape(1, 3, 1, 1)
    std = np.asarray(std, dtype=np.float32).reshape(1, 3, 1, 1)

    x = (x - mean) / std
    return torch.from_numpy(x)


# load torchvision models
try:
    from torchvision.models import (
        efficientnet_b0, EfficientNet_B0_Weights,
        convnext_tiny, ConvNeXt_Tiny_Weights
    )
except Exception as e:
    raise RuntimeError(f"torchvision model import failed : {repr(e)}")


# efficientnet-b0
eff_weights = EfficientNet_B0_Weights.DEFAULT
eff_model = efficientnet_b0(weights=eff_weights)

class EffB0Embedder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.features = model.features
        self.avgpool = model.avgpool

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)  # [B,1280]
        return x

eff_embedder = EffB0Embedder(eff_model).to(DEVICE)
eff_embedder.eval()

EFF_MEAN = np.array(eff_weights.transforms().mean, dtype=np.float32)
EFF_STD = np.array(eff_weights.transforms().std, dtype=np.float32)

# convnext-tiny
convnext_weights = ConvNeXt_Tiny_Weights.DEFAULT
convnext_model = convnext_tiny(weights=convnext_weights)

class ConvNeXtTinyEmbedder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.features = model.features
        self.avgpool = model.avgpool

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)  # [B,768]
        return x

convnext_embedder = ConvNeXtTinyEmbedder(convnext_model).to(DEVICE)
convnext_embedder.eval()

CONVNEXT_MEAN = np.array(convnext_weights.transforms().mean, dtype=np.float32)
CONVNEXT_STD = np.array(convnext_weights.transforms().std, dtype=np.float32)

# quick output dim check
with torch.no_grad():
    dummy = torch.zeros((2, 3, IMG_SIZE, IMG_SIZE), device=DEVICE)
    print("EfficientNet output :", tuple(eff_embedder(dummy).shape))
    print("ConvNeXt output     :", tuple(convnext_embedder(dummy).shape))

In [ ]:
def write_meta_if_needed(out_dir: Path, meta: dict):
    meta_path = out_dir / "meta.json"
    meta_path.write_text(json.dumps(meta, indent=2))
    return meta_path


def verify_one_feature_file(feature_path: Path, expected_frames: int, expected_dim: int):
    if not feature_path.exists():
        return False, "missing"

    try:
        arr = np.load(feature_path, mmap_mode="r")
    except Exception as e:
        return False, f"load_failed : {repr(e)}"

    if arr.shape != (expected_frames, expected_dim):
        return False, f"bad_shape : {arr.shape} expected {(expected_frames, expected_dim)}"

    return True, "ok"


def cache_video_embeddings(
    video_id: str,
    video_path: Path,
    frame_count: int,
    out_dir: Path,
    embedder,
    mean,
    std,
    embedding_dim: int,
    batch_size: int = 64,
    resize_hw=(224, 224),
    device=DEVICE
):
    out_path = out_dir / f"{video_id}.npy"

    # if already valid, skip
    ok_existing, msg_existing = verify_one_feature_file(out_path, frame_count, embedding_dim)
    if ok_existing:
        print(f"[skip] {video_id} already exists and looks valid")
        return True

    if not CHOLEC80_AVAILABLE:
        print(f"[skip] {video_id} cannot be rebuilt because Cholec80 is not mounted")
        return False

    cap = safe_open_video(video_path)
    feats = np.zeros((frame_count, embedding_dim), dtype=np.float32)

    idx = 0
    buf = []

    try:
        while idx < frame_count:
            ok, fr = cap.read()
            if not ok or fr is None:
                cap.release()
                print(f"[fail] {video_id} read failed at frame {idx}/{frame_count}")
                return False

            fr = cv2.resize(fr, resize_hw, interpolation=cv2.INTER_AREA)
            buf.append(fr)
            idx += 1

            if len(buf) == batch_size or idx == frame_count:
                batch_bgr = np.stack(buf, axis=0)
                x = bgr_batch_to_torch_rgb(batch_bgr, mean=mean, std=std).to(device)

                with torch.no_grad():
                    emb = embedder(x).detach().cpu().numpy().astype(np.float32)

                start = idx - len(buf)
                feats[start:idx] = emb
                buf = []

        cap.release()
        np.save(out_path, feats)
        return True

    except Exception as e:
        cap.release()
        print(f"[fail] {video_id} exception : {repr(e)}")
        return False

In [ ]:
# save metadata next to each cache folder

# write meta files
eff_meta = {
    "backbone": "torchvision.efficientnet_b0",
    "weights": "EfficientNet_B0_Weights.DEFAULT",
    "img_size": IMG_SIZE,
    "fps": 25,
    "normalization": {
        "mean": EFF_MEAN.tolist(),
        "std": EFF_STD.tolist()
    },
    "embedding_dim": 1280,
    "note": "Frame embeddings from EfficientNet-B0 avgpool output."
}

convnext_meta = {
    "backbone": "torchvision.convnext_tiny",
    "weights": "ConvNeXt_Tiny_Weights.DEFAULT",
    "img_size": IMG_SIZE,
    "fps": 25,
    "normalization": {
        "mean": CONVNEXT_MEAN.tolist(),
        "std": CONVNEXT_STD.tolist()
    },
    "embedding_dim": 768,
    "note": "Frame embeddings from ConvNeXt-Tiny avgpool output."
}

print("Wrote :", write_meta_if_needed(EFF_DIR, eff_meta))
print("Wrote :", write_meta_if_needed(CONVNEXT_DIR, convnext_meta))


def ensure_embeddings(backbone_name: str, out_dir: Path, embedder, mean, std, embedding_dim: int):
    out_dir.mkdir(parents=True, exist_ok=True)

    valid_existing = 0
    needs_build = []

    for r in videos_df.itertuples(index=False):
        vid = r.video_id
        nframes = int(r.frame_count)
        feature_path = out_dir / f"{vid}.npy"

        ok_existing, _ = verify_one_feature_file(feature_path, nframes, embedding_dim)

        if ok_existing:
            valid_existing += 1
        else:
            needs_build.append((vid, Path(r.path), nframes))

    if valid_existing == len(videos_df):
        summary = {
            "backbone": backbone_name,
            "embedding_dir": str(out_dir),
            "valid_existing": valid_existing,
            "rebuilt": 0,
            "failed": [],
            "elapsed_minutes": 0.0
        }

        out_summary = out_dir / "build_summary.json"
        out_summary.write_text(json.dumps(summary, indent=2))

        print(f"\n[{backbone_name}] all embeddings already exist and look valid. skipping rebuild.")
        print(json.dumps(summary, indent=2))
        return summary

    if not CHOLEC80_AVAILABLE:
        raise RuntimeError(
            f"[{backbone_name}] some embeddings are missing/invalid, but Cholec80 is not mounted so they cannot be rebuilt."
        )

    print(f"\n[{backbone_name}] {valid_existing} valid files already exist.")
    print(f"[{backbone_name}] need to build {len(needs_build)} file(s).")

    fail_list = []
    rebuilt_count = 0
    t0 = time.time()

    for vid, vp, nframes in tqdm(needs_build, desc=f"{backbone_name} embedding build"):
        ok = cache_video_embeddings(
            video_id=vid,
            video_path=vp,
            frame_count=nframes,
            out_dir=out_dir,
            embedder=embedder,
            mean=mean,
            std=std,
            embedding_dim=embedding_dim,
            batch_size=BATCH_SIZE,
            resize_hw=(IMG_SIZE, IMG_SIZE),
            device=DEVICE
        )

        if ok:
            rebuilt_count += 1
        else:
            fail_list.append(vid)

    dt = time.time() - t0

    summary = {
        "backbone": backbone_name,
        "embedding_dir": str(out_dir),
        "valid_existing": valid_existing,
        "rebuilt": rebuilt_count,
        "failed": fail_list,
        "elapsed_minutes": dt / 60.0
    }

    out_summary = out_dir / "build_summary.json"
    out_summary.write_text(json.dumps(summary, indent=2))

    print(f"\n[{backbone_name}] summary")
    print(json.dumps(summary, indent=2))

    return summary


eff_summary = ensure_embeddings(
    backbone_name="effb0",
    out_dir=EFF_DIR,
    embedder=eff_embedder,
    mean=EFF_MEAN,
    std=EFF_STD,
    embedding_dim=1280
)

convnext_summary = ensure_embeddings(
    backbone_name="convnext_tiny",
    out_dir=CONVNEXT_DIR,
    embedder=convnext_embedder,
    mean=CONVNEXT_MEAN,
    std=CONVNEXT_STD,
    embedding_dim=768
)

In [ ]:
# final cache check

def verify_backbone_folder(backbone_name: str, out_dir: Path, expected_dim: int):
    missing = []
    badshape = []

    for r in videos_df.itertuples(index=False):
        vid = r.video_id
        nframes = int(r.frame_count)
        p = out_dir / f"{vid}.npy"

        ok, msg = verify_one_feature_file(p, nframes, expected_dim)
        if not ok:
            if msg == "missing":
                missing.append(vid)
            else:
                badshape.append((vid, msg))

    print(f"\nVerification for {backbone_name}")
    print("Missing    :", len(missing))
    print("Bad shapes :", len(badshape))
    print("OK files   :", len(videos_df) - len(missing) - len(badshape))

    if len(missing) > 0:
        pd.DataFrame({"video_id": missing}).to_csv(
            ART_LOGS / f"stage2_missing_{backbone_name}.csv", index=False
        )

    if len(badshape) > 0:
        pd.DataFrame(badshape, columns=["video_id", "issue"]).to_csv(
            ART_LOGS / f"stage2_badshape_{backbone_name}.csv", index=False
        )

    return missing, badshape


eff_missing, eff_bad = verify_backbone_folder("effb0", EFF_DIR, 1280)
conv_missing, conv_bad = verify_backbone_folder("convnext_tiny", CONVNEXT_DIR, 768)

# summary table
feature_summary_df = pd.DataFrame([
    {
        "backbone": "effb0",
        "feature_dir": str(EFF_DIR),
        "expected_dim": 1280,
        "missing_files": len(eff_missing),
        "badshape_files": len(eff_bad),
        "ok_files": len(videos_df) - len(eff_missing) - len(eff_bad)
    },
    {
        "backbone": "convnext_tiny",
        "feature_dir": str(CONVNEXT_DIR),
        "expected_dim": 768,
        "missing_files": len(conv_missing),
        "badshape_files": len(conv_bad),
        "ok_files": len(videos_df) - len(conv_missing) - len(conv_bad)
    }
])

feature_summary_path = ART_COMMON / "feature_cache_summary.csv"
feature_summary_df.to_csv(feature_summary_path, index=False)

print("\nSaved feature cache summary :", feature_summary_path)
display(feature_summary_df)

# simple plot 
plt.figure(figsize=(7, 4), dpi=120)
plt.bar(feature_summary_df["backbone"], feature_summary_df["ok_files"], edgecolor="black", linewidth=0.8)
plt.title("Valid cached feature files by backbone", pad=12, fontweight="bold")
plt.xlabel("backbone")
plt.ylabel("number of valid feature files")
plt.tight_layout()
plt.savefig(ART_FIGS / "stage2_feature_cache_ok_files.png", dpi=220, bbox_inches="tight")
plt.show()
plt.close()

assert len(eff_missing) == 0 and len(eff_bad) == 0, "EfficientNet feature cache is incomplete or invalid"
assert len(conv_missing) == 0 and len(conv_bad) == 0, "ConvNeXt feature cache is incomplete or invalid"

print("\nStage 2 complete.")